# 05 - SharePoint: remote retrieval without an indexer

Query SharePoint through a **remote SharePoint knowledge source** and a knowledge base, using direct REST calls and no Vision skills. Search stores no copy of the content: no index, indexer or skillset, and no custom app registration.

| What runs | Where content lives | App registration |
|---|---|---|
| Remote SharePoint knowledge source -> knowledge base retrieve -> Copilot Retrieval API | SharePoint | None |

Retrieval evaluates the **signed-in user's** permissions in Microsoft 365. It is not a replacement for testing your source's permission model. Notebook 04 covers the indexed alternative; section 3 here compares the two.

**Run order:** run sections 1-2. Cleanup is opt-in.

## Prerequisites and safe test content

- Use the workshop tenant and a dedicated **non-sensitive test SharePoint site**, not a production library. Sign in with `az login --tenant <tenant-id>`.
- Your user needs Search Service Contributor (to create the knowledge source/base) and Search Index Data Reader or Contributor. The workshop grant script supplies Contributor.
- Remote queries require a Microsoft 365 Copilot add-on license for **each querying user**, or enabled Retrieval API pay-as-you-go billing for that user. Pay-as-you-go enablement requires at least one Copilot license in the tenant. It is separate from Search/Foundry billing.
- Upload the synthetic documents in `data/search` to the test site's default Documents library as Word documents: a general onboarding document and a restricted travel-policy document. Avoid real employee or customer data.
- Add `SHAREPOINT_SITE_URL` and a unique `WORKSHOP_PREFIX` to your untracked `.env`. Never paste access tokens or client secrets into cells, output, or chat.

These are preview APIs. Read the linked documentation again before delivering the class.

In [ ]:
import json
import os
import re
from pathlib import Path
from urllib.parse import urlparse
from uuid import UUID

import requests
from azure.identity import AzureCliCredential, get_bearer_token_provider
from dotenv import load_dotenv

root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
if not (root / 'infra' / 'main.bicep').is_file():
    raise RuntimeError('Open from the workshop root or notebooks directory.')
load_dotenv(root / '.env')
load_dotenv(root / '.env.azure')
required = ('AZURE_TENANT_ID', 'AZURE_SEARCH_ENDPOINT', 'SHAREPOINT_SITE_URL')
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise RuntimeError(f'Missing settings: {missing}. See .env.example.')
tenant_id = str(UUID(os.environ['AZURE_TENANT_ID']))
prefix = os.getenv('WORKSHOP_PREFIX') or input('Unique attendee prefix (e.g. alice01): ').strip()
if not re.fullmatch(r'[a-z][a-z0-9]{2,19}', prefix):
    raise ValueError('Use 3-20 lowercase letters/digits, starting with a letter.')
site_url = os.environ['SHAREPOINT_SITE_URL'].rstrip('/')
site = urlparse(site_url)
if (site.scheme != 'https' or not site.hostname or not site.hostname.endswith('.sharepoint.com')
        or site.username or site.password or site.port not in (None, 443)
        or site.query or site.fragment or any(c in site_url for c in ';"\r\n')):
    raise ValueError('Use a public-cloud SharePoint HTTPS site URL, not a file or sharing-link URL.')
search_endpoint = os.environ['AZURE_SEARCH_ENDPOINT'].rstrip('/')
api_version = '2026-08-01-preview'
remote_source_name = f'{prefix}-sp-remote-source'
remote_base_name = f'{prefix}-sp-remote-kb'
credential = AzureCliCredential(tenant_id=tenant_id)
search_token = get_bearer_token_provider(credential, 'https://search.azure.com/.default')
print('Search:', search_endpoint)
print('SharePoint site:', site_url)
print('API version:', api_version)


## 2. Create the remote knowledge source, knowledge base and query

No ingestion app, permissions or federated credential are required. This path creates only a remote knowledge source and knowledge base in Search. It invokes the Copilot Retrieval API with the querying user's identity; Search does not run Text Split or copy documents into an index.

We use **minimal reasoning effort + extractive data**: no LLM query planning or answer synthesis, and no Foundry model/RBAC changes are needed. A `semantic` intent passes the query directly to retrieval. This API term does not create an embedding model or Vision skill.

The `Path` KQL filter selects the test site. **It is not an authorization boundary**: invalid KQL can be ignored by the upstream Retrieval API, so inspect returned `webUrl` values. Source permissions remain the actual boundary. Results depend on Microsoft 365 indexing, supported file formats and license/billing eligibility.

In [ ]:
remote_source = {
    'name': remote_source_name, 'kind': 'remoteSharePoint',
    'description': 'Workshop SharePoint retrieval without copying content into Search.',
    'remoteSharePointParameters': {
        'filterExpression': f'Path:"{site_url}"',
        'resourceMetadata': ['Author', 'Title'],
    },
}
response = requests.put(
    f'{search_endpoint}/knowledgesources/{remote_source_name}?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json=remote_source, timeout=(10, 90),
)
response.raise_for_status()
print('Remote knowledge source ready:', remote_source_name)


In [ ]:
remote_base = {
    'name': remote_base_name,
    'description': 'Retrieve SharePoint passages without LLM planning or synthesis.',
    'knowledgeSources': [{'name': remote_source_name}],
    'models': [],
    'retrievalReasoningEffort': {'kind': 'minimal'},
    'outputMode': 'extractiveData',
}
response = requests.put(
    f'{search_endpoint}/knowledgebases/{remote_base_name}?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json=remote_base, timeout=(10, 90),
)
response.raise_for_status()
print('Extractive knowledge base ready:', remote_base_name)


In [ ]:
question = 'travel reimbursement receipts'
response = requests.post(
    f'{search_endpoint}/knowledgebases/{remote_base_name}/retrieve?api-version={api_version}',
    headers={
        'Authorization': f'Bearer {search_token()}',
        'x-ms-query-source-authorization': search_token(),
    },
    json={
        'intents': [{'type': 'semantic', 'search': question}],
        'knowledgeSourceParams': [{
            'knowledgeSourceName': remote_source_name, 'kind': 'remoteSharePoint',
            'includeReferences': True, 'includeReferenceSourceData': True,
        }],
    }, timeout=(10, 120),
)
response.raise_for_status()
remote_result = response.json()
print(json.dumps(remote_result, indent=2))
if response.status_code == 206 or any(item.get('error') for item in remote_result.get('activity', [])):
    raise RuntimeError('Retrieval was partial or a source failed. Inspect the activity output above.')
if not remote_result.get('references'):
    print('No source references returned. Check query, source permissions, M365 indexing and eligibility.')


## 3. Compare with the indexed path (notebook 04)

If you also ran notebook 04, run the same question under the same two identities on both paths. Compare indexed `url`/`chunk` results with remote references, `webUrl`, source data and activity records. Retrieval can return different passages/rankings; there is no guarantee of equal result counts or ordering.

| Capability | Indexed | Remote |
|---|---|---|
| Custom Text Split/other Search skillsets | Yes, subject to ACL-preview restrictions | No Search indexer or skillset |
| Document copy in Search | Yes | No |
| Ingestion app + federation | Required for this native-group lab | Not required |
| Permission freshness | Depends on ingestion runs and inherited-ACL resync | Microsoft 365 evaluates access; service propagation/indexing latency still applies |
| Multimodal extraction in this lab | Not enabled | Retrieval API path is textual; images/charts are not a multimodal retrieval pipeline |
| Additional eligibility | Indexer preview and tenant admin setup | Copilot license or enabled Retrieval API pay-as-you-go per user |

Remote limits include 200 requests/user/hour, 1,500 query characters and up to 25 results; verify current limits before class. A newly uploaded document might not immediately appear in remote retrieval. A syntactically accepted knowledge source does not prove user licensing, source access or successful retrieval.

**Acceptance exercise:** demonstrate a known allowed document and a known denied document for both users, including native SharePoint-group access. Do not call the permission exercise complete based only on a fluent answer, an empty response, or a successful resource PUT.

## 4. Optional Search-object cleanup

Enable the following cell to remove only this notebook's knowledge base and knowledge source. Missing objects (404) are explicitly allowed; authorization failures and other errors are not ignored. Nothing in SharePoint or Entra was created by this notebook, so there is nothing else to remove.

Clear all notebook outputs before sharing; outputs can contain private SharePoint document text and URLs even though tokens are not printed.

In [ ]:
CLEAN_UP = False
if CLEAN_UP:
    for collection, name in [('knowledgebases', remote_base_name), ('knowledgesources', remote_source_name)]:
        response = requests.delete(
            f'{search_endpoint}/{collection}/{name}?api-version={api_version}',
            headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 90),
        )
        if response.status_code != 404:
            response.raise_for_status()
        print('Deleted or already absent:', name)
else:
    print('Search-object cleanup disabled.')


## References

- [Remote SharePoint knowledge source and licensing](https://learn.microsoft.com/azure/search/agentic-knowledge-source-how-to-sharepoint-remote)
- [Create a knowledge base](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-create-knowledge-base)
- [Minimal reasoning effort](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-set-retrieval-reasoning-effort)
- [Retrieve action and semantic intents](https://learn.microsoft.com/azure/search/agentic-retrieval-how-to-retrieve)

Preview API versions and Microsoft 365 eligibility can change.